# 03 — Comparing Buffer Sizes

A single buffer answers *is this within range?*

Multiple buffers at different radii answer *how close?* — turning a binary yes/no into a gradient of proximity. This is how blast zones, noise contours, and risk maps are structured: concentric rings, each representing a different intensity level.

```
inner ring  →  high intensity   (lethal / severe)
middle ring →  medium intensity (damaging / moderate)
outer ring  →  low intensity    (warning / minor)
```

## Setup

In [1]:
import json
import math
from pathlib import Path
from ipyleaflet import Map, GeoJSON, LayersControl, basemaps

DATA_DIR = Path("data")

with open(DATA_DIR / "targets.geojson") as f:
    targets_fc = json.load(f)


def destination_point(lon, lat, bearing_deg, distance_km):
    R = 6371.0
    d = distance_km / R
    phi1 = math.radians(lat)
    lam1 = math.radians(lon)
    theta = math.radians(bearing_deg)
    phi2 = math.asin(math.sin(phi1)*math.cos(d) + math.cos(phi1)*math.sin(d)*math.cos(theta))
    lam2 = lam1 + math.atan2(math.sin(theta)*math.sin(d)*math.cos(phi1),
                              math.cos(d) - math.sin(phi1)*math.sin(phi2))
    return [math.degrees(lam2), math.degrees(phi2)]

def point_buffer(lon, lat, radius_km, n_points=64):
    ring = [destination_point(lon, lat, 360.0*i/n_points, radius_km) for i in range(n_points)]
    ring.append(ring[0])
    return {"type": "Polygon", "coordinates": [ring]}

def buffer_feature(lon, lat, radius_km, name="", n_points=64):
    return {
        "type": "Feature",
        "properties": {"name": name, "radius_km": radius_km},
        "geometry": point_buffer(lon, lat, radius_km, n_points)
    }

# Helper: extract lon/lat from a Point feature
def target_coords(name):
    feat = next(f for f in targets_fc["features"] if f["properties"]["name"] == name)
    return feat["geometry"]["coordinates"]

print(f"Loaded {len(targets_fc['features'])} targets.")

Loaded 4 targets.


## Concentric Rings Around One Target

Three radii around the same center produces the nested ring structure. The outer ring contains the inner rings — the area covered by the inner ring is always a subset of the outer ring.

In [2]:
tehran = target_coords("Tehran")

# Three impact zones — radii in km
zones = [
    {"radius_km": 50,  "label": "Lethal",    "color": "#e74c3c"},
    {"radius_km": 150, "label": "Damaging",  "color": "#e67e22"},
    {"radius_km": 300, "label": "Warning",   "color": "#f1c40f"},
]

m = Map(center=(tehran[1], tehran[0]), zoom=4, basemap=basemaps.CartoDB.Positron)

# Add outermost first so inner rings render on top
for zone in reversed(zones):
    feat = buffer_feature(*tehran, radius_km=zone["radius_km"], name=zone["label"])
    m.add(GeoJSON(
        data={"type": "FeatureCollection", "features": [feat]},
        name=f"{zone['label']} ({zone['radius_km']} km)",
        style={"color": zone["color"], "fillColor": zone["color"],
               "fillOpacity": 0.35, "weight": 1.5}
    ))

m.add(LayersControl())

print("Zone radii:", [z['radius_km'] for z in zones], "km")
m

Zone radii: [50, 150, 300] km


Map(center=[35.695, 51.388], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom…

## Interpreting Zone Overlap

The inner zone is entirely contained within the outer zones. The **area of the outer ring that is not in the inner ring** represents a region of *less* intensity — close enough to be affected, but not in the core zone.

For a given target, we can compute the exclusive area of each band (annular ring area) using the formula for a circle:

```
band area = π × r_outer² − π × r_inner²
```

In [3]:
import math

radii = [0] + [z["radius_km"] for z in zones]   # add 0 as inner bound of first band

print(f"{'Band':<25} {'Exclusive area':>18}")
print("─" * 46)
for i in range(1, len(radii)):
    r_inner = radii[i - 1]
    r_outer = radii[i]
    area_km2 = math.pi * (r_outer**2 - r_inner**2)
    label = f"{zones[i-1]['label']} zone (0–{r_outer} km)"
    print(f"  {label:<23}  {area_km2:>14,.0f} km²")

Band                          Exclusive area
──────────────────────────────────────────────
  Lethal zone (0–50 km)             7,854 km²
  Damaging zone (0–150 km)          62,832 km²
  Warning zone (0–300 km)         212,058 km²


## Overlapping Buffers from Multiple Targets

When two targets are close enough, their buffers overlap. The overlap region is reachable from both — useful for identifying shared risk zones or mutual coverage.

In [4]:
# Compare Tehran and Riyadh — roughly 1,400 km apart
riyadh = target_coords("Riyadh")

multi_target_fc = {
    "type": "FeatureCollection",
    "features": [
        buffer_feature(*tehran, radius_km=500, name="Tehran 500 km"),
        buffer_feature(*riyadh, radius_km=500, name="Riyadh 500 km"),
    ]
}

# Point features for the targets
point_fc = {
    "type": "FeatureCollection",
    "features": [
        {"type": "Feature", "properties": {"name": "Tehran"},
         "geometry": {"type": "Point", "coordinates": tehran}},
        {"type": "Feature", "properties": {"name": "Riyadh"},
         "geometry": {"type": "Point", "coordinates": riyadh}},
    ]
}

m2 = Map(center=(35, 47), zoom=4, basemap=basemaps.CartoDB.Positron)
m2.add(GeoJSON(
    data=multi_target_fc,
    style={"color": "#8e44ad", "fillColor": "#8e44ad", "fillOpacity": 0.25, "weight": 1.5}
))
m2.add(GeoJSON(
    data=point_fc,
    style={"color": "#2c3e50", "fillColor": "#2c3e50", "fillOpacity": 0.8, "weight": 1}
))
m2

Map(center=[35, 47], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_tex…

## Exercise A

Create **three concentric buffers** (75 km, 200 km, 450 km) around **Honolulu** (`[-157.855, 21.305]`).

1. Display all three on a single map with distinct colors.
2. Print the exclusive band area (km²) for each ring.
3. Which band has the largest area, and why?

In [5]:
import math
from ipyleaflet import Map, GeoJSON, basemaps

honolulu = [-157.855, 21.305]

# 1. Three concentric buffers (75, 200, 450 km)
# 2. Print band areas
# 3. Identify and explain the largest band
# Your code here

honolulu_zones = [
    {"radius_km": 75, "label": "Core", "color": "#e74c3c"},
    {"radius_km": 200, "label": "Middle", "color": "#f39c12"},
    {"radius_km": 450, "label": "Outer", "color": "#3498db"},
]

m = Map(center=(honolulu[1], honolulu[0]), zoom=5, basemap=basemaps.CartoDB.Positron)

# Add outermost first so smaller buffers stay visible on top.
for zone in reversed(honolulu_zones):
    feat = buffer_feature(
        *honolulu,
        radius_km=zone["radius_km"],
        name=zone["label"]
    )

    m.add(GeoJSON(
        data={"type": "FeatureCollection", "features": [feat]},
        name=f"Honolulu {zone['label']} ({zone['radius_km']} km)",
        style={
            "color": zone["color"],
            "fillColor": zone["color"],
            "fillOpacity": 0.30,
            "weight": 1.5,
        },
    ))

m.add(LayersControl())

radii = [0] + [zone["radius_km"] for zone in honolulu_zones]
band_areas = []

print(f"{'Band':<18} {'Radius range':<16} {'Exclusive area':>18}")
print("-" * 56)

for i, zone in enumerate(honolulu_zones, start=1):
    r_inner = radii[i - 1]
    r_outer = radii[i]
    area_km2 = math.pi * (r_outer**2 - r_inner**2)

    band_areas.append((zone["label"], area_km2))

    print(
        f"{zone['label']:<18} "
        f"{r_inner:>3}-{r_outer:<10} km "
        f"{area_km2:>14,.0f} km²"
    )

largest_band, largest_area = max(band_areas, key=lambda item: item[1])

print()
print(f"The {largest_band} band has the largest exclusive area ({largest_area:,.0f} km²).")
print("It is largest because area grows with radius squared, so the outer annulus adds much more area than the inner bands.")

m

Band               Radius range         Exclusive area
--------------------------------------------------------
Core                 0-75         km         17,671 km²
Middle              75-200        km        107,992 km²
Outer              200-450        km        510,509 km²

The Outer band has the largest exclusive area (510,509 km²).
It is largest because area grows with radius squared, so the outer annulus adds much more area than the inner bands.


Map(center=[21.305, -157.855], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zo…

## Exercise B

Create a 400 km buffer around **each** of the four targets (Tehran, Honolulu, Madrid, Riyadh) and display them all on a world map.

1. Use `LayersControl` so each target's buffer can be toggled.
2. Identify by inspection which pairs of buffers overlap.
3. Print the haversine distance between the two closest targets to confirm whether they should overlap.

In [6]:
import math
from ipyleaflet import Map, GeoJSON, LayersControl, basemaps

def haversine_km(lon1, lat1, lon2, lat2):
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return R * 2 * math.asin(math.sqrt(a))

# 1. 400 km buffer around each target with LayersControl
# 2. Identify overlapping pairs by inspection
# 3. Print distances between all pairs, confirm which overlap at 400 km
# Your code here
targets = [
    {"name": "Tehran", "coords": target_coords("Tehran"), "color": "#e74c3c"},
    {"name": "Honolulu", "coords": [-157.855, 21.305], "color": "#3498db"},
    {"name": "Madrid", "coords": target_coords("Madrid"), "color": "#2ecc71"},
    {"name": "Riyadh", "coords": target_coords("Riyadh"), "color": "#9b59b6"},
]

radius_km = 400

m = Map(center=(25, 20), zoom=2, basemap=basemaps.CartoDB.Positron)


for target in targets:
    lon, lat = target["coords"]

    feat = buffer_feature(
        lon,
        lat,
        radius_km=radius_km,
        name=f"{target['name']} {radius_km} km"
    )

    m.add(GeoJSON(
        data={"type": "FeatureCollection", "features": [feat]},
        name=f"{target['name']} buffer",
        style={
            "color": target["color"],
            "fillColor": target["color"],
            "fillOpacity": 0.25,
            "weight": 1.5,
        },
    ))

m.add(LayersControl())


closest_pair = None
closest_distance = float("inf")

print(f"{'Pair':<24} {'Distance':>12} {'Overlap?':>12}")
print("-" * 50)

for i in range(len(targets)):
    for j in range(i + 1, len(targets)):
        a = targets[i]
        b = targets[j]

        lon1, lat1 = a["coords"]
        lon2, lat2 = b["coords"]

        distance = haversine_km(lon1, lat1, lon2, lat2)
        overlap = distance <= radius_km * 2

        if distance < closest_distance:
            closest_distance = distance
            closest_pair = (a["name"], b["name"])

        pair_name = f"{a['name']} - {b['name']}"
        print(f"{pair_name:<24} {distance:>10.0f} km {str(overlap):>12}")

print()
print(
    f"The closest pair is {closest_pair[0]} and {closest_pair[1]} "
    f"at about {closest_distance:.0f} km apart."
)

if closest_distance <= radius_km * 2:
    print("Their 400 km buffers should overlap because the distance is less than 800 km.")
else:
    print("Their 400 km buffers should not overlap because the distance is greater than 800 km.")

m

Pair                         Distance     Overlap?
--------------------------------------------------
Tehran - Honolulu             12969 km        False
Tehran - Madrid                4774 km        False
Tehran - Riyadh                1305 km        False
Honolulu - Madrid             12649 km        False
Honolulu - Riyadh             14255 km        False
Madrid - Riyadh                4960 km        False

The closest pair is Tehran and Riyadh at about 1305 km apart.
Their 400 km buffers should not overlap because the distance is greater than 800 km.


Map(center=[25, 20], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_tex…

---

## Check Your Understanding

**1.** What does overlap between two buffers from different targets represent in a real-world context?

**2.** Why do we compare multiple buffer sizes rather than picking one radius and using it for everything?

```python
# No code needed — answer in your own words
1. Overlap between two buffers from different targets means that the same area is within the chosen distance of both targets. In a real-world context, that overlap could represent a shared risk zone, shared coverage area, or a place affected by both sources.

2. We compare multiple buffer sizes because one radius only gives a yes/no answer: inside or outside. Multiple radii show different levels of distance or intensity, such as high-risk, medium-risk, and low-risk zones. This gives a more useful picture than treating every location inside one large buffer as the same.
```

## Next

In [04 — Buffer Visualization Strategies](./04-Buffer_Visualization_Strategies.ipynb), we focus on how to style and order buffer layers so the map communicates clearly — avoiding the common mistakes that hide results or mislead interpretation.